# Finance Data Tutorial

Notebook ngắn này hướng dẫn 3 bước:
1. Bật cập nhật `financial` trong config updater.
2. Chạy updater để lưu parquet theo từng symbol.
3. Query dữ liệu tài chính bằng DuckDB để lọc cổ phiếu.

In [1]:
from vnstock_forecast.config import load_config
from vnstock_forecast.engine.schemas.config import to_app_config
from vnstock_forecast.engine.data.updater import update
from vnstock_forecast.engine.data.query import query_financial, query_sql, list_financial_metrics, query_financial_by_statement, search_financial_metrics
from vnstock_forecast.config import list_symbols_lists, query_symbols_list, search_symbols_lists

## 1) Bật financial updater bằng runtime override

Cấu trúc mới của updater:
- `data.updater.ohlcv.*`
- `data.updater.financial.*`

In [2]:
cfg = load_config(overrides=[
    "data.updater.financial.update=true",
    "data.updater.financial.symbols=[VHM,VNM,FPT]",
    "data.updater.ohlcv.update=true",
])
app_cfg = to_app_config(cfg)
print(app_cfg.data.updater)

UpdaterConfig(ohlcv=OhlcvUpdaterConfig(update=True, client=<DataClient.vietstock: 'vietstock'>, symbols=['ACB', 'BID', 'CTG', 'DGC', 'FPT', 'GAS', 'GVR', 'HDB', 'HPG', 'LPB', 'MBB', 'MSN', 'MWG', 'PLX', 'SAB', 'SHB', 'SSB', 'SSI', 'STB', 'TCB', 'TPB', 'VCB', 'VHM', 'VIB', 'VIC', 'VJC', 'VNM', 'VPB', 'VPL', 'VRE'], resolutions=['D'], lookback_days=3600), financial=FinancialUpdaterConfig(update=True, client=<DataClient.vietcap: 'vietcap'>, symbols=['VHM', 'VNM', 'FPT']))


## 2) Chạy updater

Khi chạy thành công, dữ liệu được lưu theo cấu trúc:
- `data/finance/<SYMBOL>/cash_flow.parquet`
- `data/finance/<SYMBOL>/income_statement.parquet`
- `data/finance/<SYMBOL>/balance_sheet.parquet`
- `data/finance/<SYMBOL>/footnote.parquet`
- `data/finance/<SYMBOL>/statistics.parquet`

In [ ]:
ok = update(app_cfg.data.updater)
print("Update status:", ok)

## 3) Query tài chính để lọc cổ phiếu

`query_financial` trả dữ liệu long-format: `symbol, statement, metric, description, period, value`.

In [ ]:
df = query_financial(
    symbols=["VHM", "VNM", "FPT"],
    statements="statistics",
    metrics=["roe", "roa"],
    periods="Q4_2025",
    min_value=0.1,
)
df.head(20)

In [ ]:
# Ví dụ SQL screening: lấy mã có ROE Q4_2025 > 0.15
screen_df = query_sql(
    """
    SELECT symbol, value AS roe
    FROM finance_long
    WHERE statement = 'statistics'
      AND metric = 'roe'
      AND period = 'Q4_2025'
      AND value > 0.15
    ORDER BY roe DESC
    """
)
screen_df

In [17]:
search_financial_metrics("loan", "balance_sheet")

,statement,metric,description
0,balance_sheet,allowance_for_balances_with_and_loans_to_other...,Dự phòng rủi ro
1,balance_sheet,deposits_and_loans_from_other_credit_institutions,Tiền gửi và vay các Tổ chức tín dụng khác
2,balance_sheet,deposits_and_loans_from_other_credit_institutions,Tiền gửi của các tổ chức tín dụng khác
3,balance_sheet,due_to_gov_and_loans_from_sbv,Các khoản nợ chính phủ và NHNN Việt Nam
4,balance_sheet,irrevocable_loan_commitments,Cam kết cho vay không hủy ngang
5,balance_sheet,less:_provision_for_losses_on_loans_and_advanc...,Dự phòng rủi ro cho vay khách hàng
6,balance_sheet,loans,Các khoản cho vay
7,balance_sheet,loans_and_advances_to_customers,Cho vay khách hàng
8,balance_sheet,loans_and_advances_to_customers,Cho vay và ứng trước cho khách hàng
9,balance_sheet,loans_and_advances_to_customers,CHO VAY VÀ ỨNG TRƯỚC CHO KHÁCH HÀNG


In [ ]:
list_financial_metrics()

,statement,metric,description
0,balance_sheet,accounts_receivable,Các khoản phải thu
1,balance_sheet,accrued_expenses,Chi phí phải trả
2,balance_sheet,accumulated_depreciation,Khấu hao lũy kế TSCĐ vô hình
3,balance_sheet,accumulated_depreciation,Khấu hao lũy kế tài sản đầu tư
4,balance_sheet,accumulated_depreciation,Khấu hao lũy kế tài sản thuê tài chính
...,...,...,...
362,statistics,quickratio,NaN
363,statistics,roa,NaN
364,statistics,roe,NaN
365,statistics,roic,NaN


In [19]:
search_symbols_lists("finan")

['consumer_finance_l4_8773',
 'financial_services_l2_8700',
 'financial_services_l3_8770',
 'financials_l1_8000',
 'specialty_finance_l4_8775']

In [35]:
expression = "pe < 10 and (pb < 1 or roe > 0.15) and footnote.short-term_loans != 0"
df = query_financial_by_statement(symbols=query_symbols_list("vn100"), statement=expression)

In [36]:
df

,symbol,statistics.pe,statistics.pb,statistics.roe,footnote.short-term_loans
0,ACB,7.709243,1.274386,0.175577,NaN
1,ANV,6.403363,1.814647,0.316112,1.548828e+12
2,BID,9.527614,1.698460,0.191402,NaN
3,BMP,9.227163,3.940193,0.440473,5.490000e+10
4,BWE,9.475870,1.570954,0.175187,8.649473e+11
5,CTG,7.634319,1.479959,0.212451,NaN
6,CTS,9.927901,2.007430,0.226050,8.422953e+12
7,DBC,6.002495,1.121015,0.203163,4.770218e+12
8,DGC,9.717407,1.951974,0.212899,1.546200e+12
9,HAG,9.063925,1.458023,0.193601,5.039320e+12
